[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/05_workflows.ipynb)

# 05 · Workflows — reproducible, templated research

**Workflows** turn a research process into a reusable **template** you can run
again and again — over different companies, dates, or universes — and get a
consistent, structured report each time. It's the general-purpose engine behind
deep-research reports on **[Bigdata.com](https://bigdata.com)**.

**What it's good for**
- Standardised analyses you re-run regularly (e.g. a weekly company brief)
- Running the same prompt across a whole watchlist or universe
- Going from a saved template → a single API call in your own systems

`POST /v1/workflow/execute` (on **`agents.bigdata.com`**) streams results as SSE,
just like the Research Agent. You can run an **inline** template or a **stored
template by ID**.

In [1]:
import os
import requests

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = os.environ["BIGDATA_API_KEY"]

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

### SSE helper
Same streaming idea as notebook 04; here the event payload is under `delta`.

In [2]:
import json

def run_workflow(template, inputs=None, model_name="base", time_range=None, show_progress=True):
    """Execute a workflow. `template` is an inline dict or a stored template ID (str)."""
    payload = {"template": template, "model_name": model_name}
    if inputs:
        payload["input"] = inputs
    if time_range:
        payload["time_range"] = time_range

    answer_parts = []
    with requests.post(f"{AGENTS_BASE}/v1/workflow/execute", headers=HEADERS, json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[len("data:"):].strip())
            delta = event.get("delta", {})
            mtype = delta.get("type")
            if mtype == "ERROR":
                raise RuntimeError(delta.get("error"))
            if mtype == "ACTION" and show_progress:
                print("  action:", delta.get("tool_name"))
            elif mtype == "ANSWER":
                answer_parts.append(delta.get("content", ""))
    return "".join(answer_parts)

### Example 1 — Run an inline template
Define a template with a `{{ placeholder }}` in the prompt and declare it in
`expected_input`. Supply values via `input`. (Declaring `expected_input` is
required whenever you pass `input`.)

In [3]:
template = {
    "name": "One-line company summary",
    "prompt": "In one sentence, summarise what {{ company }} does and its main business segments.",
    "expected_input": {"company": {"type": "string"}},
}

answer = run_workflow(template, inputs={"company": "Apple"})
print("\n=== RESULT ===\n")
print(answer)

  action: search_companies


  action: get_company_tearsheet



=== RESULT ===

Apple Inc. is a global technology corporation that specializes in the design, manufacture, and sale of electronic devices and digital services, with its business activities organized into primary reporting segments including iPhone, Mac, iPad, Wearables/Home/Accessories, and Services .


### Example 2 — Browse and reuse stored templates
List your templates (and community templates) and run one by its ID. Templates
let you save a vetted research process once and call it from code.

In [4]:
templates = requests.get(f"{AGENTS_BASE}/v1/workflow/templates", headers=HEADERS, timeout=30).json()["results"]
print(f"You have {len(templates)} saved template(s). First few:")
for t in templates[:5]:
    print("  -", t["name"])

# To run a stored template, pass its ID as the `template` argument and provide
# any inputs it declares, e.g.:
#   answer = run_workflow("<template_id>", inputs={"company_list": "Apple, Microsoft"})

You have 20 saved template(s). First few:
  - Middle Eastern Drone Companies — Financial Intelligence Matrix
  - Bottom-Up Sentiment Analysis — Universe Recipe
  - US Pure-Play Space Sector — Fundamental Deep Dive
  - Daily Positive/Negative News Narrative — Company with Date Range v3
  - Daily Positive/Negative News Narrative — Company with Date Range V2


## From the app to the API
You can design and save a workflow visually, then call it by ID from your own
systems — the saved template *is* the contract. Explore and build multi-agent
workflows in **Agent Swarm**: 👉 https://agent-swarm.labs.bigdata.com/

---

## Real-world use cases & demo apps
These showcase what teams build on top of these APIs. _(Links to be added.)_

- **Build-your-own Workflow** (deep-research report generator) — _TODO: add link_
- **Portfolio Commentary / Rebalance** — _TODO: add link_
- **Monitoring Briefs** (briefs pipeline & examples) — _TODO: add link_
- **Thematic Screener** (mind-map generator, smart batching, full pipeline) — _TODO: add link_
- **Risk Analyzer** — _TODO: add link_

**Connection / integration cookbooks** (popular with technical teams):
- Agent → Search API — _TODO: add link_
- Agent → Research Agent API — _TODO: add link_
- Agent → Bigdata MCP — _TODO: add link_

_Common patterns: fundamental teams favour build-your-own workflows, automated
portfolio commentary, and monitoring briefs; quant teams favour the thematic
screener and risk analyzer._

_Source: [Bigdata.com](https://bigdata.com) · Workflows docs: https://docs.bigdata.com/api-reference/workflows/execute-workflow_